In [9]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    return results



In [ ]:
# generated_dataset = generate_dataset()

# with open("generated_dataset.json", "w") as f:
#     json.dump(generated_dataset, f, indent=2)


In [ ]:
for test_case in generate_dataset:
    print(f"this is the test case: {test_case}")


this is the test case: {'task': 'Write a Python function that extracts the AWS account ID from an ARN string. The function should take an ARN as input and return the 12-digit account ID.'}
this is the test case: {'task': 'Create a JSON configuration object for an AWS Lambda function that includes runtime environment as Python 3.11, timeout of 60 seconds, memory of 512 MB, and a VPC configuration with two subnet IDs and one security group ID.'}
this is the test case: {'task': 'Write a regex pattern that validates an AWS S3 bucket name. The pattern should ensure the bucket name is 3-63 characters long, contains only lowercase letters, numbers, hyphens, and periods, does not start or end with a hyphen or period, and does not contain consecutive periods.'}


In [15]:
with open("generated_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [13]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS IAM Role ARN Validator\n\nHere's a comprehensive solution with multiple approaches:\n\n```python\nimport re\n\ndef validate_iam_role_arn(arn: str) -> bool:\n    \"\"\"\n    Validates an AWS IAM role ARN format.\n    \n    Expected format: arn:aws:iam::ACCOUNT_ID:role/ROLE_NAME\n    \n    Args:\n        arn: The ARN string to validate\n        \n    Returns:\n        True if the ARN is valid, False otherwise\n    \"\"\"\n    if not isinstance(arn, str):\n        return False\n    \n    # AWS IAM Role ARN pattern\n    pattern = r'^arn:aws:iam::\\d{12}:role/[a-zA-Z0-9+=,.@\\-_/]+$'\n    \n    return bool(re.match(pattern, arn))\n\n\ndef validate_iam_role_arn_detailed(arn: str) -> dict:\n    \"\"\"\n    Validates an AWS IAM role ARN and returns detailed information.\n    \n    Args:\n        arn: The ARN string to validate\n        \n    Returns:\n        Dictionary with validation result and details\n    \"\"\"\n    if not isinstance(arn, str):\n        return {